In [1]:
import pandas as pd

df = pd.read_csv('../data/cleaned/superstore_cleaned.csv')

# Key numbers
total_revenue = round(df['Sales'].sum(), 2)
total_profit = round(df['Profit'].sum(), 2)
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
avg_order_value = round(df['Sales'].mean(), 2)

print("=" * 50)
print("RETAILLENS — KEY BUSINESS METRICS")
print("=" * 50)
print(f"Total Revenue     : ${total_revenue:,}")
print(f"Total Profit      : ${total_profit:,}")
print(f"Total Orders      : {total_orders:,}")
print(f"Total Customers   : {total_customers:,}")
print(f"Avg Order Value   : ${avg_order_value:,}")
print("=" * 50)

RETAILLENS — KEY BUSINESS METRICS
Total Revenue     : $2,297,200.86
Total Profit      : $286,397.02
Total Orders      : 5,009
Total Customers   : 793
Avg Order Value   : $229.86


In [2]:
cat = df.groupby('Category')[['Sales', 'Profit']].sum()
cat['Margin %'] = round((cat['Profit'] / cat['Sales']) * 100, 2)

print("=" * 50)
print("CATEGORY PERFORMANCE")
print("=" * 50)
print(cat.to_string())
print()
print("INSIGHT: Technology has highest profit margin")
print("INSIGHT: Furniture has high sales but lowest margin")

CATEGORY PERFORMANCE
                       Sales       Profit  Margin %
Category                                           
Furniture        741999.7953   18451.2728      2.49
Office Supplies  719047.0320  122490.8008     17.04
Technology       836154.0330  145454.9481     17.40

INSIGHT: Technology has highest profit margin
INSIGHT: Furniture has high sales but lowest margin


In [3]:
low_discount = df[df['Discount'] <= 0.1]['Profit'].mean()
high_discount = df[df['Discount'] > 0.2]['Profit'].mean()

print("=" * 50)
print("DISCOUNT IMPACT ANALYSIS")
print("=" * 50)
print(f"Avg Profit with LOW discount  : ${round(low_discount, 2)}")
print(f"Avg Profit with HIGH discount : ${round(high_discount, 2)}")
print()
print("INSIGHT: High discounts (>20%) cause average losses")
print("RECOMMENDATION: Cap all discounts at 15%")

DISCOUNT IMPACT ANALYSIS
Avg Profit with LOW discount  : $67.46
Avg Profit with HIGH discount : $-97.18

INSIGHT: High discounts (>20%) cause average losses
RECOMMENDATION: Cap all discounts at 15%


In [4]:
region = df.groupby('Region')[['Sales', 'Profit']].sum()
region['Margin %'] = round((region['Profit'] / region['Sales']) * 100, 2)
region = region.sort_values('Sales', ascending=False)

print("=" * 50)
print("REGION WISE PERFORMANCE")
print("=" * 50)
print(region.to_string())
print()
best_region = region['Sales'].idxmax()
worst_region = region['Margin %'].idxmin()
print(f"INSIGHT: {best_region} region has highest revenue")
print(f"INSIGHT: {worst_region} region has lowest profit margin")

REGION WISE PERFORMANCE
               Sales       Profit  Margin %
Region                                     
West     725457.8245  108418.4489     14.94
East     678781.2400   91522.7800     13.48
Central  501239.8908   39706.3625      7.92
South    391721.9050   46749.4303     11.93

INSIGHT: West region has highest revenue
INSIGHT: Central region has lowest profit margin


In [5]:
customer = df.groupby('Customer ID')['Order ID'].nunique()
repeat = (customer > 1).sum()
one_time = (customer == 1).sum()
top_customers = df.groupby('Customer Name')['Sales'].sum().nlargest(5)

print("=" * 50)
print("CUSTOMER INSIGHTS")
print("=" * 50)
print(f"Repeat Customers  : {repeat}")
print(f"One-time Customers: {one_time}")
print(f"Retention Rate    : {round(repeat/(repeat+one_time)*100, 1)}%")
print()
print("TOP 5 CUSTOMERS BY REVENUE:")
print(top_customers.to_string())
print()
print("INSIGHT: Focus on converting one-time buyers to repeat customers")

CUSTOMER INSIGHTS
Repeat Customers  : 781
One-time Customers: 12
Retention Rate    : 98.5%

TOP 5 CUSTOMERS BY REVENUE:
Customer Name
Sean Miller      25043.050
Tamara Chand     19052.218
Raymond Buch     15117.339
Tom Ashbrook     14595.620
Adrian Barton    14473.571

INSIGHT: Focus on converting one-time buyers to repeat customers


In [6]:
loss_products = df.groupby('Product Name')['Profit'].sum()
loss_products = loss_products[loss_products < 0].sort_values().head(10)

total_loss = round(loss_products.sum(), 2)

print("=" * 50)
print("TOP 10 LOSS MAKING PRODUCTS")
print("=" * 50)
print(loss_products.to_string())
print()
print(f"Total Loss from these products: ${total_loss:,}")
print("RECOMMENDATION: Review pricing or discontinue these products")

TOP 10 LOSS MAKING PRODUCTS
Product Name
Cubify CubeX 3D Printer Double Head Print                           -8879.9704
Lexmark MX611dhe Monochrome Laser Printer                           -4589.9730
Cubify CubeX 3D Printer Triple Head Print                           -3839.9904
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases            -2876.1156
Bush Advantage Collection Racetrack Conference Table                -1934.3976
GBC DocuBind P400 Electric Binding System                           -1878.1662
Cisco TelePresence System EX90 Videoconferencing Unit               -1811.0784
Martin Yale Chadless Opener Electric Letter Opener                  -1299.1836
Balt Solid Wood Round Tables                                        -1201.0581
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables   -1148.4375

Total Loss from these products: $-29,458.37
RECOMMENDATION: Review pricing or discontinue these products


In [7]:
shipping = df.groupby('Ship Mode').agg(
    avg_days=('Shipping Days', 'mean'),
    total_orders=('Order ID', 'count')
).round(1)

print("=" * 50)
print("SHIPPING PERFORMANCE")
print("=" * 50)
print(shipping.to_string())
print()
most_used = shipping['total_orders'].idxmax()
print(f"INSIGHT: {most_used} is most used shipping mode")
print("RECOMMENDATION: Promote faster shipping options to premium customers")

SHIPPING PERFORMANCE
                avg_days  total_orders
Ship Mode                             
First Class          2.2          1538
Same Day             0.0           543
Second Class         3.2          1945
Standard Class       5.0          5968

INSIGHT: Standard Class is most used shipping mode
RECOMMENDATION: Promote faster shipping options to premium customers


In [9]:
print("=" * 50)
print("RETAILLENS — EXECUTIVE SUMMARY")
print("=" * 50)
print(f"""
1. REVENUE
   Total Revenue: ${total_revenue:,}
   Total Profit : ${total_profit:,}

2. BIGGEST PROBLEM
   High discounts (>20%) are causing losses
   Recommend: Cap discounts at 15%

3. BEST PERFORMING
   Technology category has best profit margin
   West region leads in revenue

4. CUSTOMER HEALTH
   Retention rate needs improvement
   Focus on repeat customer campaigns

5. QUICK WINS
   Discontinue top 10 loss-making products
   Push faster shipping to premium customers
""")

RETAILLENS — EXECUTIVE SUMMARY

1. REVENUE
   Total Revenue: $2,297,200.86
   Total Profit : $286,397.02

2. BIGGEST PROBLEM
   High discounts (>20%) are causing losses
   Recommend: Cap discounts at 15%

3. BEST PERFORMING
   Technology category has best profit margin
   West region leads in revenue

4. CUSTOMER HEALTH
   Retention rate needs improvement
   Focus on repeat customer campaigns

5. QUICK WINS
   Discontinue top 10 loss-making products
   Push faster shipping to premium customers

